# 07 — Experiment 5: Explainability (Grad-CAM + SHAP + LIME)
Generates class-discriminative visual explanations for the final model's predictions using
three complementary methods:

- **Grad-CAM** — gradient-based, shows *which MRI regions* drove the prediction
- **SHAP** — additive feature attribution, shows *how much* each region contributed, per class
- **LIME** — model-agnostic superpixel perturbation, a sanity-check independent of gradients

**[our contribution]**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src import config as cfg
from src.data_loader import get_filepaths_and_labels
from src.preprocessing import load_and_preprocess_image
from src.models.fusion_model import build_model

# Load the best fine-tuned model from notebook 06 (Experiment 4)
ckpt_path = os.path.join(cfg.CHECKPOINTS_DIR, 'cbam_transformer_ssl.keras')
model = tf.keras.models.load_model(ckpt_path)
print("Loaded:", ckpt_path)

In [ ]:
# Grab one test image to explain
test_paths, test_labels = get_filepaths_and_labels('test')
idx = 0
img = load_and_preprocess_image(tf.constant(test_paths[idx]))
img_batch = img[tf.newaxis, ...]
true_label = cfg.CLASS_NAMES[test_labels[idx]]
print("True label:", true_label)

In [ ]:
# --- Grad-CAM ---
from src.explainability.gradcam import make_gradcam_heatmap, overlay_heatmap_on_image

heatmap, pred_idx = make_gradcam_heatmap(img_batch.numpy(), model)
print("Predicted:", cfg.CLASS_NAMES[pred_idx])

overlay = overlay_heatmap_on_image(img.numpy().squeeze(), heatmap)
plt.figure(figsize=(4, 4))
plt.imshow(overlay)
plt.title(f"Grad-CAM — pred: {cfg.CLASS_NAMES[pred_idx]}, true: {true_label}")
plt.axis('off')
plt.show()

In [ ]:
# --- SHAP ---
from src.explainability.shap_explain import build_explainer, explain_instance, plot_shap_summary

# Small background set for the baseline distribution (use a handful of training images)
train_paths, _ = get_filepaths_and_labels('train')
background = np.stack([
    load_and_preprocess_image(tf.constant(p)).numpy() for p in train_paths[:30]
])

explainer = build_explainer(model, background)
shap_values = explain_instance(explainer, img_batch.numpy(), num_classes=cfg.NUM_CLASSES)
plot_shap_summary(shap_values, img_batch.numpy(), cfg.CLASS_NAMES,
                   out_path=os.path.join(cfg.RESULTS_DIR, 'shap_example.png'))

In [ ]:
# --- LIME ---
from src.explainability.lime_explain import explain_instance as lime_explain, get_explanation_overlay

explanation = lime_explain(model, img.numpy().squeeze(), cfg.CLASS_NAMES, num_samples=1000)
overlay = get_explanation_overlay(explanation, pred_idx)

plt.figure(figsize=(4, 4))
plt.imshow(overlay)
plt.title(f"LIME — pred: {cfg.CLASS_NAMES[pred_idx]}")
plt.axis('off')
plt.show()

Run this for several test images (loop over more indices) and look for consistency: do
Grad-CAM/SHAP/LIME agree on which brain regions matter for e.g. 'ModerateDemented' predictions?
Consistent, anatomically-plausible regions (e.g. hippocampal/medial temporal areas known to
atrophy in AD) are what makes the explainability claim credible in a thesis writeup — random or
scattered attributions across methods would be worth flagging, not hiding.